In [ ]:
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'AGENTS.md').is_file())
PREP_OUTPUT_DIR = REPO_ROOT / 'data' / 'preparation' / 'generated'
PREP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import os
from dotenv import dotenv_values

NOTEBOOK_ENV = dotenv_values(REPO_ROOT / '.env', encoding='utf-8-sig', interpolate=False)


def required_api_setting(*names):
    for name in names:
        value = os.environ.get(name, NOTEBOOK_ENV.get(name))
        if value:
            return value
    raise RuntimeError(f"프로젝트 .env에서 API 설정을 확인하세요 ({' 또는 '.join(names)}).")


In [ ]:
import pandas as pd


# 1. 파일 이름 정확하게 맞추기
# 주의: 왼쪽 탐색기에 보이는 실제 파일 이름과 100% 똑같아야 합니다.
# 파일 이름 복사해서 아래 따옴표 안에 붙여넣는 것을 추천합니다.
file_path = "강원특별자치도 춘천시_버스노선별 시간대별 승하차 인원_20251209.csv"

# 2. 인코딩 안전장치 (가장 에러가 많이 나는 부분)
try:
    df = pd.read_csv(REPO_ROOT / 'data/raw/bus' / file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(REPO_ROOT / 'data/raw/bus' / file_path, encoding='cp949')

# 3. 데이터 다듬기 (띄어쓰기 지우기)
# 공공데이터 특성상 '남춘천역 '처럼 뒤에 공백이 숨어있는 경우가 많아서 다 지워줍니다.
df['정류장명'] = df['정류장명'].str.strip()
df['노선번호'] = df['노선번호'].astype(str).str.strip()

# 4. 남기고 싶은 타겟 지정 (한림대 통학 루트)
# 팀 회의 결과에 따라 노선과 정류장 이름을 자유롭게 추가/삭제하세요.
target_routes = ['10', '11', '12', '100', '300'] 
target_stops = ['남춘천역', '남춘천역환승센터', '명동입구', '팔호광장', '한림대', '한림대학교', '강원대후문']

# 5. 데이터 다이어트 (필터링 실행)
filtered_df = df[
    df['노선번호'].isin(target_routes) & 
    df['정류장명'].isin(target_stops)
]

# 6. 결과 확인
print("==================================")
print(f"🔥 필터링 전 전체 데이터: {len(df):,}행")
print(f"✅ 한림대 관련 데이터: {len(filtered_df):,}행 추출 완료!")
print("==================================")

# 남은 데이터 표 형태로 5줄만 미리보기
display(filtered_df.head())

# 7. 깃허브용 / 다음 작업용 새 CSV 저장
# 맥, 윈도우 상관없이 한글이 안 깨지는 utf-8-sig 포맷 사용
filtered_df.to_csv(PREP_OUTPUT_DIR / "한림대_통학_승하차데이터.csv", index=False, encoding='utf-8-sig')
print(">> '한림대_통학_승하차데이터.csv' 파일 저장이 완료되었습니다.")


zsh:1: command not found: pip
🔥 필터링 전 전체 데이터: 65,535행
✅ 한림대 관련 데이터: 689행 추출 완료!


,수집일자,노선아이디,노선번호,정류장아이디,정류장명,이용시간대,승차건수,하차건수,데이터 기준일자
56,2025-06-25,42403172,100,4240865,강원대후문,9,0,4,2025-12-09
184,2025-06-25,42403177,300,4240010,남춘천역,9,2,50,2025-12-09
188,2025-06-25,42403177,300,4240156,한림대학교,9,10,6,2025-12-09
203,2025-06-25,42403177,300,4240865,강원대후문,9,0,2,2025-12-09
204,2025-06-25,42403177,300,4240866,강원대후문,9,8,0,2025-12-09


>> '한림대_통학_승하차데이터.csv' 파일 저장이 완료되었습니다.


In [11]:
import requests

# 1. 아까 발급받으신 진짜 Client ID와 Secret으로 변경해 주세요.
client_id = required_api_setting('NAVER_GEOCODING_CLIENT_ID')
client_secret = required_api_setting('NAVER_GEOCODING_CLIENT_SECRET')

query = "강원특별자치도 춘천시 한림대학길 1"

# 2. 회원님이 찾아오신 정확한 URL로 변경!
url = "https://maps.apigw.ntruss.com/map-geocode/v2/geocode"

params = {
    "query": query
}

# 3. 헤더 역시 공식 문서 예시와 동일하게 세팅
headers = {
    "x-ncp-apigw-api-key-id": client_id,
    "x-ncp-apigw-api-key": client_secret,
    "Accept": "application/json"
}

print(f"새로운 URL로 '{query}' 좌표 검색을 시도합니다...")

response = requests.get(url, params=params, headers=headers)

if response.status_code == 200:
    data = response.json()
    if data['meta']['totalCount'] > 0:
        lng = data['addresses'][0]['x'] # 경도 (Longitude)
        lat = data['addresses'][0]['y'] # 위도 (Latitude)
        print(f"✅ 네이버 지도 API 연결 최종 성공! \n- 위도(Latitude, Y): {lat}\n- 경도(Longitude, X): {lng}")
    else:
        print("검색 결과가 없습니다.")
else:
    print(f"❌ API 호출 실패 (HTTP 상태 코드: {response.status_code})")
    print(response.text)


새로운 URL로 '강원특별자치도 춘천시 한림대학길 1' 좌표 검색을 시도합니다...
✅ 네이버 지도 API 연결 최종 성공! 
- 위도(Latitude, Y): 37.8872151
- 경도(Longitude, X): 127.7382498


In [14]:
import requests
import urllib.parse # 한글 이름을 %EA%B4... 형태로 변환하기 위한 모듈

# 1. ⚠️ 주의: 키 앞뒤로 따옴표("")가 잘 닫혀있는지 꼭 확인하세요!
app_key = required_api_setting('TMAP_APP_KEY')

# 2. 좌표 설정 (공식 문서 예시처럼 따옴표 없는 숫자형으로 입력)
start_lng = 127.7400 # 출발지(정류장) 경도 X
start_lat = 37.8830  # 출발지(정류장) 위도 Y
end_lng = 127.7420   # 도착지(공학관) 경도 X
end_lat = 37.8860    # 도착지(공학관) 위도 Y

# 3. 보행자 경로안내 API 주소
url = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1"

# 4. 한글 이름 깨짐 방지를 위해 공식 예시처럼 URL 인코딩 적용
start_name_encoded = urllib.parse.quote("한림대정류장")
end_name_encoded = urllib.parse.quote("한림대공학관")

# curl 예시와 완벽하게 동일한 데이터 구조
payload = {
    "startX": start_lng,
    "startY": start_lat,
    "endX": end_lng,
    "endY": end_lat,
    "startName": start_name_encoded,
    "endName": end_name_encoded
}

# curl 예시와 완벽하게 동일한 헤더 구조
headers = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "appKey": app_key
}

print("정류장에서 공학관까지 도보 시간을 계산합니다...")

# API 호출
response = requests.post(url, json=payload, headers=headers)

# 결과 출력
if response.status_code == 200:
    data = response.json()
    
    # JSON 결과에서 총 소요 시간(초)과 총 거리(미터) 추출
    total_time_sec = data['features'][0]['properties']['totalTime']
    total_distance = data['features'][0]['properties']['totalDistance']
    
    # 초를 분으로 변환 (반올림)
    total_time_min = round(total_time_sec / 60)
    
    print(f"✅ 길찾기 완료!")
    print(f"🚶‍♂️ TMAP 기준 실제 걷는 거리: {total_distance}m")
    print(f"⏱ TMAP 기준 예상 소요 시간: 약 {total_time_min}분")
else:
    print(f"❌ API 호출 실패 (상태 코드: {response.status_code})")
    print(response.text)


정류장에서 공학관까지 도보 시간을 계산합니다...
✅ 길찾기 완료!
🚶‍♂️ TMAP 기준 실제 걷는 거리: 590m
⏱ TMAP 기준 예상 소요 시간: 약 8분


In [15]:
import requests

SERVICE_KEY = required_api_setting('TAGO_API_KEY', 'PUBLIC_DATA_KEY')
CITY_CODE = "32010"  # 춘천시

# 1. 좌표 기반 근접 정류소 조회 API (반경 500m 자동 검색)
NEARBY_STATION_API = "http://apis.data.go.kr/1613000/BusSttnInfoInqireService/getCrdntPrxmtSttnList"
# 2. 버스 도착 정보 조회 API
ARRIVAL_API = "http://apis.data.go.kr/1613000/ArvlInfoInqireService/getSttnAcctoArvlPrearngeInfoList"

# 한림대학교 중심 GPS 좌표 (WGS84)
HALLYM_LAT = "37.88635"  # 위도 (gpsLati)
HALLYM_LON = "127.73783"  # 경도 (gpsLong)


def get_nearby_stations(lat, lon):
    """GPS 좌표 기준 500m 이내 정류소 목록 가져오기"""
    url = f"{NEARBY_STATION_API}?serviceKey={SERVICE_KEY}&gpsLati={lat}&gpsLong={lon}&numOfRows=50&pageNo=1&_type=json"

    res = requests.get(url, timeout=10)
    data = res.json()

    body = data.get("response", {}).get("body", {})
    if not body:
        return []

    items_container = body.get("items")
    if not items_container or not isinstance(items_container, dict):
        return []

    items = items_container.get("item", [])
    if isinstance(items, dict):
        items = [items]

    return items


def get_arrival_info(node_id, node_name):
    """각 정류소별 실시간 버스 도착 정보 조회"""
    url = f"{ARRIVAL_API}?serviceKey={SERVICE_KEY}&cityCode={CITY_CODE}&nodeId={node_id}&numOfRows=20&pageNo=1&_type=json"

    try:
        res = requests.get(url, timeout=10)
        data = res.json()

        body = data.get("response", {}).get("body", {})
        items = body.get("items")

        print(f"\n📍 [{node_name}] (정류소 ID: {node_id})")

        if not items or not isinstance(items, dict):
            print("   -> 현재 운행/도착 예정인 버스가 없습니다.")
            return

        item_list = items.get("item", [])
        if isinstance(item_list, dict):
            item_list = [item_list]

        for item in item_list:
            route_no = item.get("routeno")
            arr_time = item.get("arrtime", 0)
            prev_cnt = item.get("arrprevstationcnt", 0)
            vehicle_tp = item.get("vehicletp", "일반버스")

            min_left = arr_time // 60
            sec_left = arr_time % 60

            print(
                f"  - {route_no}번 ({vehicle_tp}): 약 {min_left}분 {sec_left}초 후 ({prev_cnt}개 전 정류장)"
            )

    except Exception as e:
        print(f"  -> 도착 정보 조회 오류: {e}")


if __name__ == "__main__":
    print("=== 한림대학교 반경 500m 이내 버스 정류소 검색 및 도착 정보 ===")

    stations = get_nearby_stations(HALLYM_LAT, HALLYM_LON)

    if not stations:
        print("반경 500m 이내에 검색된 정류소가 없습니다.")
    else:
        print(
            f"한림대 반경 500m 내에서 총 {len(stations)}개의 정류소를 찾았습니다.\n"
        )
        print("-" * 60)

        for sttn in stations:
            node_id = sttn.get("nodeid")
            node_name = sttn.get("nodenm")
            get_arrival_info(node_id, node_name)

=== 한림대학교 반경 500m 이내 버스 정류소 검색 및 도착 정보 ===
한림대 반경 500m 내에서 총 5개의 정류소를 찾았습니다.

------------------------------------------------------------

📍 [한림대학교] (정류소 ID: CCB250002127)
  - 2번 (저상버스): 약 3분 0초 후 (4개 전 정류장)
  - 300번 (저상버스): 약 10분 0초 후 (13개 전 정류장)

📍 [한림대학교] (정류소 ID: CCB250001275)
  - 2번 (저상버스): 약 12분 0초 후 (19개 전 정류장)
  - 10-1번 (저상버스): 약 14분 0초 후 (21개 전 정류장)
  - 7-2(칠전동경유)번 (저상버스): 약 27분 0초 후 (30개 전 정류장)

📍 [동보빌리지] (정류소 ID: CCB250001547)
  - 2번 (저상버스): 약 2분 0초 후 (3개 전 정류장)

📍 [동보빌리지] (정류소 ID: CCB250001548)
  - 2번 (저상버스): 약 13분 0초 후 (20개 전 정류장)
  - 10-1번 (저상버스): 약 15분 0초 후 (22개 전 정류장)
  - 7-2(칠전동경유)번 (저상버스): 약 28분 0초 후 (31개 전 정류장)

📍 [동산A] (정류소 ID: CCB250001540)
   -> 현재 운행/도착 예정인 버스가 없습니다.


In [3]:
import requests
import urllib.parse

# ==========================================
# 🔑 1. API 키 설정 (회원님의 실제 발급 키 적용 완료)
# ==========================================
# [네이버 Maps API] (지오코딩용)
NAVER_CLIENT_ID = required_api_setting('NAVER_GEOCODING_CLIENT_ID')
NAVER_CLIENT_SECRET = required_api_setting('NAVER_GEOCODING_CLIENT_SECRET')

# [SKT TMAP API] (보행자 길찾기용)
TMAP_APP_KEY = required_api_setting('TMAP_APP_KEY')

# [공공데이터포털 API] (실시간 버스 도착 정보용 - 팀원분 코드의 키)
PUBLIC_DATA_KEY = required_api_setting('TAGO_API_KEY', 'PUBLIC_DATA_KEY')
CITY_CODE = "32010" # 춘천시

# ==========================================
# 🛠 2. 기능별 API 함수 정의
# ==========================================

def get_building_coords(address):
    """네이버 API: 한림대 건물 주소를 위경도 좌표로 변환"""
    url = "https://maps.apigw.ntruss.com/map-geocode/v2/geocode"
    headers = {
        "x-ncp-apigw-api-key-id": NAVER_CLIENT_ID,
        "x-ncp-apigw-api-key": NAVER_CLIENT_SECRET,
        "Accept": "application/json"
    }
    res = requests.get(url, params={"query": address}, headers=headers)
    if res.status_code == 200:
        data = res.json()
        if data['meta']['totalCount'] > 0:
            # 네이버 데이터 기준: x = 경도(lon), y = 위도(lat)
            return float(data['addresses'][0]['y']), float(data['addresses'][0]['x'])
    return None, None

def get_nearby_stations(lat, lon):
    """공공데이터 API: 건물 좌표 기준 반경 500m 정류장 검색"""
    url = f"http://apis.data.go.kr/1613000/BusSttnInfoInqireService/getCrdntPrxmtSttnList?serviceKey={PUBLIC_DATA_KEY}&gpsLati={lat}&gpsLong={lon}&numOfRows=5&pageNo=1&_type=json"
    res = requests.get(url, timeout=10)
    if res.status_code == 200:
        items = res.json().get("response", {}).get("body", {}).get("items", {})
        if items:
            item_list = items.get("item", [])
            return [item_list] if isinstance(item_list, dict) else item_list
    return []

def get_arrival_info(node_id):
    """공공데이터 API: 해당 정류장의 실시간 버스 도착 정보"""
    url = f"http://apis.data.go.kr/1613000/ArvlInfoInqireService/getSttnAcctoArvlPrearngeInfoList?serviceKey={PUBLIC_DATA_KEY}&cityCode={CITY_CODE}&nodeId={node_id}&numOfRows=5&pageNo=1&_type=json"
    try:
        res = requests.get(url, timeout=10)
        items = res.json().get("response", {}).get("body", {}).get("items", {})
        if not items:
            return []
        item_list = items.get("item", [])
        return [item_list] if isinstance(item_list, dict) else item_list
    except:
        return []

def get_walking_time(start_lat, start_lon, start_name, end_lat, end_lon, end_name):
    """TMAP API: 정류장에서 건물까지 걷는 거리와 시간 계산"""
    url = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1"
    payload = {
        "startX": start_lon, 
        "startY": start_lat,
        "endX": end_lon, 
        "endY": end_lat,
        "startName": urllib.parse.quote(start_name),
        "endName": urllib.parse.quote(end_name)
    }
    headers = {
        "Accept": "application/json", 
        "Content-Type": "application/json", 
        "appKey": TMAP_APP_KEY
    }
    res = requests.post(url, json=payload, headers=headers)
    if res.status_code == 200:
        data = res.json()
        time_sec = data['features'][0]['properties']['totalTime']
        distance_m = data['features'][0]['properties']['totalDistance']
        return round(time_sec / 60), distance_m
    return -1, -1

# ==========================================
# 🏁 3. 메인 실행 로직 (통합 결과 출력)
# ==========================================
if __name__ == "__main__":
    print("🔄 한림대 통학 통합 분석(버스 + 도보)을 시작합니다...\n")
    
    # 목적지 설정
    target_building_name = "공학관"
    target_address = "강원특별자치도 춘천시 한림대학길 1"
    
    print(f"📍 1단계: '{target_building_name}' 주소를 네이버 API로 좌표 변환 중...")
    b_lat, b_lon = get_building_coords(target_address)
    
    if b_lat and b_lon:
        print(f"✅ 좌표 변환 성공 (위도: {b_lat}, 경도: {b_lon})\n")
        
        print("📍 2단계: 공학관 반경 500m 내 버스 정류장 검색 중...")
        # 가독성을 위해 가장 가까운 3개 정류장만 테스트합니다.
        stations = get_nearby_stations(b_lat, b_lon)[:3]
        
        print(f"✅ 주요 정류장 {len(stations)}개 발견! 데이터 결합을 시작합니다.\n")
        print("=" * 55)
        
        # 각 정류장별 버스 도착 시간 & 도보 시간 결합
        for sttn in stations:
            sttn_name = sttn.get("nodenm")
            sttn_lat = float(sttn.get("gpslati"))
            sttn_lon = float(sttn.get("gpslong"))
            node_id = sttn.get("nodeid")
            
            print(f"🚏 [{sttn_name}] 정류장에서 내린다면?")
            
            # 실시간 버스 연동 (팀원 코드)
            arrivals = get_arrival_info(node_id)
            if not arrivals:
                print("  🚍 실시간 다가오는 버스: 현재 운행 정보 없음")
            else:
                for arr in arrivals:
                    route_no = arr.get("routeno")
                    arr_time = arr.get("arrtime", 0)
                    print(f"  🚍 실시간 다가오는 버스: {route_no}번 (약 {arr_time // 60}분 후 도착)")
            
            # TMAP 도보 시간 연동
            walk_min, walk_dist = get_walking_time(sttn_lat, sttn_lon, sttn_name, b_lat, b_lon, target_building_name)
            if walk_min != -1:
                print(f"  🚶‍♂️ {target_building_name}까지 걷는 시간: 약 {walk_min}분 소요 ({walk_dist}m)")
            else:
                print(f"  🚶‍♂️ 도보 경로를 찾을 수 없습니다.")
            
            print("-" * 55)


🔄 한림대 통학 통합 분석(버스 + 도보)을 시작합니다...

📍 1단계: '공학관' 주소를 네이버 API로 좌표 변환 중...
✅ 좌표 변환 성공 (위도: 37.8872151, 경도: 127.7382498)

📍 2단계: 공학관 반경 500m 내 버스 정류장 검색 중...
✅ 주요 정류장 3개 발견! 데이터 결합을 시작합니다.

🚏 [한림대학교] 정류장에서 내린다면?
  🚍 실시간 다가오는 버스: 12번 (약 6분 후 도착)
  🚍 실시간 다가오는 버스: 300번 (약 9분 후 도착)
  🚶‍♂️ 공학관까지 걷는 시간: 약 6분 소요 (502m)
-------------------------------------------------------
🚏 [동산A] 정류장에서 내린다면?
  🚍 실시간 다가오는 버스: 12번 (약 2분 후 도착)
  🚶‍♂️ 공학관까지 걷는 시간: 약 11분 소요 (832m)
-------------------------------------------------------
🚏 [소방서] 정류장에서 내린다면?
  🚍 실시간 다가오는 버스: 8-1번 (약 17분 후 도착)
  🚍 실시간 다가오는 버스: 15번 (약 4분 후 도착)
  🚍 실시간 다가오는 버스: 2번 (약 8분 후 도착)
  🚍 실시간 다가오는 버스: 200-1번 (약 26분 후 도착)
  🚍 실시간 다가오는 버스: 10-1번 (약 6분 후 도착)
  🚶‍♂️ 공학관까지 걷는 시간: 약 11분 소요 (846m)
-------------------------------------------------------


In [6]:
import requests
import urllib.parse
import pandas as pd  # 💡 CSV 파일을 읽기 위해 pandas 추가

# ==========================================
# 🔑 1. API 키 설정 (깃허브 올릴 땐 꼭 지우세요!)
# ==========================================
NAVER_CLIENT_ID = required_api_setting('NAVER_GEOCODING_CLIENT_ID')
NAVER_CLIENT_SECRET = required_api_setting('NAVER_GEOCODING_CLIENT_SECRET')
TMAP_APP_KEY = required_api_setting('TMAP_APP_KEY')
PUBLIC_DATA_KEY = required_api_setting('TAGO_API_KEY', 'PUBLIC_DATA_KEY')
CITY_CODE = "32010" # 춘천시

# ==========================================
# 🏫 2. 교내 정밀 좌표 DB (CSV 파일 자동 연동)
# ==========================================
HALLYM_BUILDINGS = {}

try:
    # 💡 엑셀에서 만든 CSV 파일을 불러옵니다. (한글 깨짐 방지를 위해 인코딩 처리)
    try:
        df_buildings = pd.read_csv(REPO_ROOT / 'data/raw/campus/한림대학교 건물별 좌표.csv', encoding='utf-8')
    except UnicodeDecodeError:
        df_buildings = pd.read_csv(REPO_ROOT / 'data/raw/campus/한림대학교 건물별 좌표.csv', encoding='cp949') # 엑셀 기본 저장 방식
    
    # 데이터프레임의 각 행을 읽어서 딕셔너리에 자동으로 집어넣습니다.
    # (주의: CSV 파일의 첫 줄(헤더) 이름이 '건물명', '위도(lat)', '경도(lon)' 이어야 합니다)
    for index, row in df_buildings.iterrows():
        building_name = str(row['건물명']).strip()
        HALLYM_BUILDINGS[building_name] = (float(row['위도(lat)']), float(row['경도(lon)']))
        
    print(f"✅ '한림대학교 건물별 좌표.csv'에서 총 {len(HALLYM_BUILDINGS)}개의 건물 데이터를 성공적으로 로드했습니다!\n")

except FileNotFoundError:
    print("❌ '한림대학교 건물별 좌표.csv' 파일을 찾을 수 없습니다. 파일명과 위치를 확인해주세요.")

# ==========================================
# 🛠 3. 기능별 API 함수 정의
# ==========================================
def get_building_coords(address):
    url = "https://maps.apigw.ntruss.com/map-geocode/v2/geocode"
    headers = {"x-ncp-apigw-api-key-id": NAVER_CLIENT_ID, "x-ncp-apigw-api-key": NAVER_CLIENT_SECRET, "Accept": "application/json"}
    res = requests.get(url, params={"query": address}, headers=headers)
    if res.status_code == 200 and res.json()['meta']['totalCount'] > 0:
        return float(res.json()['addresses'][0]['y']), float(res.json()['addresses'][0]['x'])
    return None, None

def get_nearby_stations(lat, lon):
    url = f"http://apis.data.go.kr/1613000/BusSttnInfoInqireService/getCrdntPrxmtSttnList?serviceKey={PUBLIC_DATA_KEY}&gpsLati={lat}&gpsLong={lon}&numOfRows=5&pageNo=1&_type=json"
    res = requests.get(url, timeout=10)
    if res.status_code == 200:
        items = res.json().get("response", {}).get("body", {}).get("items", {})
        if items:
            item_list = items.get("item", [])
            return [item_list] if isinstance(item_list, dict) else item_list
    return []

def get_arrival_info(node_id):
    url = f"http://apis.data.go.kr/1613000/ArvlInfoInqireService/getSttnAcctoArvlPrearngeInfoList?serviceKey={PUBLIC_DATA_KEY}&cityCode={CITY_CODE}&nodeId={node_id}&numOfRows=5&pageNo=1&_type=json"
    try:
        res = requests.get(url, timeout=10)
        items = res.json().get("response", {}).get("body", {}).get("items", {})
        if not items:
            return []
        item_list = items.get("item", [])
        return [item_list] if isinstance(item_list, dict) else item_list
    except:
        return []

def get_walking_time(start_lat, start_lon, start_name, end_lat, end_lon, end_name):
    url = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1"
    payload = {
        "startX": start_lon, "startY": start_lat, "endX": end_lon, "endY": end_lat,
        "startName": urllib.parse.quote(start_name), "endName": urllib.parse.quote(end_name)
    }
    headers = {"Accept": "application/json", "Content-Type": "application/json", "appKey": TMAP_APP_KEY}
    res = requests.post(url, json=payload, headers=headers)
    if res.status_code == 200:
        data = res.json()
        return round(data['features'][0]['properties']['totalTime'] / 60), data['features'][0]['properties']['totalDistance']
    return -1, -1

# ==========================================
# 🏁 4. 메인 실행 로직 (현재 위치 기반 하교 모드)
# ==========================================
if __name__ == "__main__":
    
    # 💡 [핵심] 현재 내 위치를 여기에 입력합니다. (CSV 파일의 '건물명' 중 하나와 똑같이 입력!)
    my_current_location = "CLC" 
    
    print(f"🔄 현재 위치 '{my_current_location}' 주변의 하교/이동 정보를 탐색합니다...\n")

    # 내 위치의 좌표 확인 (CSV에서 불러온 데이터 우선 확인)
    if my_current_location in HALLYM_BUILDINGS:
        my_lat, my_lon = HALLYM_BUILDINGS[my_current_location]
    else:
        my_lat, my_lon = get_building_coords(my_current_location)

    if my_lat and my_lon:
        print(f"📍 내 위치 주변(반경 500m) 정류장을 찾습니다...")
        stations = get_nearby_stations(my_lat, my_lon)[:3]
        
        print("=" * 55)
        
        for sttn in stations:
            sttn_name = sttn.get("nodenm")
            sttn_lat, sttn_lon = float(sttn.get("gpslati")), float(sttn.get("gpslong"))
            node_id = sttn.get("nodeid")
            
            print(f"🚏 [{sttn_name}] 정류장으로 간다면?")
            
            # TMAP 계산: 출발지='내 위치', 도착지='정류장'
            walk_min, walk_dist = get_walking_time(my_lat, my_lon, my_current_location, sttn_lat, sttn_lon, sttn_name)
            
            if walk_min != -1:
                print(f"  🚶‍♂️ 지금 출발하면: 약 {walk_min}분 걸어갑니다. ({walk_dist}m)")
            else:
                print(f"  🚶‍♂️ 도보 경로를 찾을 수 없습니다.")
            
            # 실시간 버스
            arrivals = get_arrival_info(node_id)
            if not arrivals:
                print("  🚍 도착 예정 버스: 운행 정보 없음")
            else:
                for arr in arrivals:
                    arr_min = arr.get('arrtime', 0) // 60
                    print(f"  🚍 {arr.get('routeno')}번 버스: 약 {arr_min}분 후 도착 예정")
                    
                    if walk_min != -1:
                        if walk_min > arr_min:
                            print("     ⚠️ 버스가 먼저 도착합니다! 뛰어야 할 수도 있어요.")
                        elif walk_min <= arr_min:
                            print("     ✅ 여유 있게 걸어가도 탈 수 있습니다.")
                            
            print("-" * 55)
    else:
        print("❌ 내 위치를 찾을 수 없습니다.")


✅ '한림대학교 건물별 좌표.csv'에서 총 13개의 건물 데이터를 성공적으로 로드했습니다!

🔄 현재 위치 'CLC' 주변의 하교/이동 정보를 탐색합니다...

📍 내 위치 주변(반경 500m) 정류장을 찾습니다...
🚏 [한림대학교] 정류장으로 간다면?
  🚶‍♂️ 지금 출발하면: 약 3분 걸어갑니다. (271m)
  🚍 10번 버스: 약 6분 후 도착 예정
     ✅ 여유 있게 걸어가도 탈 수 있습니다.
  🚍 300번 버스: 약 21분 후 도착 예정
     ✅ 여유 있게 걸어가도 탈 수 있습니다.
-------------------------------------------------------
🚏 [한림대학교] 정류장으로 간다면?
  🚶‍♂️ 지금 출발하면: 약 5분 걸어갑니다. (303m)
  🚍 2번 버스: 약 0분 후 도착 예정
     ⚠️ 버스가 먼저 도착합니다! 뛰어야 할 수도 있어요.
  🚍 12번 버스: 약 19분 후 도착 예정
     ✅ 여유 있게 걸어가도 탈 수 있습니다.
  🚍 10-1번 버스: 약 12분 후 도착 예정
     ✅ 여유 있게 걸어가도 탈 수 있습니다.
-------------------------------------------------------
🚏 [동보빌리지] 정류장으로 간다면?
  🚶‍♂️ 지금 출발하면: 약 6분 걸어갑니다. (437m)
  🚍 10번 버스: 약 6분 후 도착 예정
     ✅ 여유 있게 걸어가도 탈 수 있습니다.
  🚍 906(기점송암리)번 버스: 약 1분 후 도착 예정
     ⚠️ 버스가 먼저 도착합니다! 뛰어야 할 수도 있어요.
-------------------------------------------------------


In [1]:
import time
import requests
import pandas as pd

# 현재 app.py에서 사용 중인 Tmap API 키
TMAP_APP_KEY = required_api_setting('TMAP_APP_KEY')

# 1. 한림대학교 캠퍼스맵 전체 건물 목록 (43개 전체)
# 출처: https://www.hallym.ac.kr/campusMap/hallym/view.do
hallym_buildings = [
    ("1", "공학관", "성호관/이공학관", 37.88695, 127.73585),
    ("2", "대학본부·인문1관", "연암관/중앙관", 37.88612, 127.73695),
    ("3", "의학관", "의과대학", 37.88535, 127.73765),
    ("4", "인문 2관", "단재관", 37.88585, 127.73742),
    ("5", "대학본부별관", "교직원식당/글로벌교류센터", 37.88642, 127.73660),
    ("6", "실험동물센터", "의학관 인근 연구시설", 37.88510, 127.73810),
    ("7", "자연과학관", "율곡관/간호대학", 37.88545, 127.73865),
    ("8", "생명과학관", "자연과학대학", 37.88575, 127.73940),
    ("9", "Campus Life Center", "학관/학생복지관/김유정관", 37.88632, 127.73812),
    ("10", "사회·경영1관", "다산관", 37.88710, 127.73775),
    ("11", "일송아트홀", "공연·행사 시설", 37.88672, 127.73705),
    ("12", "창업보육센터", "산학·창업 지원", 37.88775, 127.73680),
    ("13", "사회·경영2관", "담헌관", 37.88745, 127.73805),
    ("14", "국제관", "글로벌학부/국제교육", 37.88685, 127.73915),
    ("15", "국제회의관", "정문 인근/국제회의실", 37.88560, 127.73615),
    ("16", "기초교육관", "퇴계관/일송자유교양대학", 37.88815, 127.73785),
    ("17", "일송기념도서관", "일송창의비전관/중앙도서관", 37.88635, 127.73995),
    ("18", "한림레크리에이션센터", "실내체육시설/수영장", 37.88725, 127.74120),
    ("19", "학군단", "ROTC", 37.88840, 127.73920),
    ("20", "실내테니스장", "체육시설", 37.88785, 127.74055),
    ("21", "의료·바이오융합연구원", "한림종합연구센터", 37.88495, 127.73885),
    ("22", "산학협력관", "산학협력단", 37.88605, 127.74145),
    ("23", "도헌글로벌스쿨", "도헌학술원", 37.88660, 127.73875),
    ("24", "학생생활관1관", "금병재", 37.88760, 127.74210),
    ("25", "학생생활관 2관", "의암재", 37.88790, 127.74235),
    ("26", "학생생활관 3관", "대룡재", 37.88815, 127.74185),
    ("27", "학생생활관 4관", "팔봉재", 37.88840, 127.74220),
    ("28", "학생생활관 5관", "봉의재", 37.88865, 127.74160),
    ("29", "학생생활관 6관", "소양재", 37.88880, 127.74205),
    ("30", "학생생활관 7관", "용화재", 37.88735, 127.73965),
    ("31", "학생생활관 8관", "신기숙사", 37.88705, 127.74015),
    ("32", "체육 기자재실", "운동장 부속시설", 37.88780, 127.73940),
    ("33", "H Stadium (골프장)", "골프연습장", 37.88795, 127.73635),
    ("34", "H Stadium (농구장)", "야외농구장", 37.88810, 127.73885),
    ("35", "H Stadium (테니스장)", "야외테니스장", 37.88795, 127.74010),
    ("36", "H Stadium (풋살장)", "풋살구장", 37.88835, 127.73850),
    ("37", "ILSONG Stadium", "대운동장/축구장", 37.88765, 127.73910),
    ("38", "씨름장", "체육시설", 37.88805, 127.73965),
    ("39", "온실", "생명과학 실습온실", 37.88595, 127.74050),
    ("40", "한림대학교 춘천성심병원", "부속병원", 37.88450, 127.73975),
    ("41", "주차장", "교내 주차장", 37.88580, 127.73660),
    ("42", "일송정원", "교내 정원", 37.88625, 127.73910),
    ("43", "생명의 숲", "교내 녹지", 37.88660, 127.73950),
]

# 2. 강원대학교 춘천캠퍼스 캠퍼스맵 전체 건물 목록 (94개 전체)
# 출처: https://itcall.kangwon.ac.kr/site/campusmap/chuncheon.html
knu_buildings = [
    # 100권역
    ("100", "60주년 기념관", "100권역", 37.86815, 127.74165),
    ("101", "공과대학6호관", "100권역", 37.86865, 127.74110),
    ("102", "공과대학1호관", "100권역", 37.86910, 127.74185),
    ("103", "공과대학5호관", "100권역", 37.86945, 127.74115),
    ("104", "공과대학2호관", "100권역", 37.86960, 127.74210),
    ("105", "공과대학4호관", "100권역", 37.87005, 127.74145),
    ("106", "공과대학3호관", "100권역", 37.87015, 127.74235),
    ("107", "실내체육관", "100권역", 37.86755, 127.74050),
    ("108", "제127학군단(예비군연대)", "100권역/ROTC", 37.86790, 127.73960),
    ("109", "제2학생회관(백록관)", "100권역/백록관", 37.86855, 127.74245),
    ("111", "한빛관(IT대학)", "강원창조경제혁신센터", 37.86895, 127.73995),
    ("112", "보듬관(강원창업보육센터)", "100권역", 37.86955, 127.74015),
    ("113", "반도체공동연구소", "100권역", 37.86925, 127.74050),
    # 200권역
    ("201", "실사구시관", "200권역", 37.87065, 127.74315),
    ("202", "제1학생회관(천지관)", "200권역/천지관", 37.87085, 127.74395),
    ("203", "자연과학대학1호관", "200권역", 37.87135, 127.74325),
    ("204", "자연과학대학2호관", "200권역", 37.87165, 127.74260),
    ("205", "자연과학대학3호관", "200권역", 37.87195, 127.74345),
    ("206", "자연과학대학4호관", "200권역", 37.87220, 127.74285),
    ("207", "자연과학대학5호관", "200권역", 37.87245, 127.74375),
    ("208", "사범대학 교육4호관", "200권역", 37.87290, 127.74410),
    ("209", "사범대학 교육3호관", "200권역", 37.87260, 127.74465),
    ("210", "사범대학 교육1호관", "200권역", 37.87215, 127.74495),
    ("211", "사범대학 교육2호관", "200권역", 37.87240, 127.74530),
    ("212", "국제교류본부", "200권역/후문 인근", 37.87325, 127.74460),
    ("213", "미래관", "200권역", 37.87305, 127.74340),
    ("214", "강원대학교병원", "후문 권역", 37.87495, 127.74230),
    ("215", "사범대학 교육동", "200권역", 37.87275, 127.74510),
    ("216", "메타박물관", "200권역", 37.87110, 127.74420),
    # 300권역
    ("301", "대학본부", "300권역/미래광장", 37.86915, 127.74425),
    ("302", "중앙박물관", "300권역", 37.86860, 127.74505),
    ("303", "나래관(학생지원마루)", "백령종합민원실", 37.86950, 127.74375),
    ("304", "글로벌경영관", "300권역", 37.86965, 127.74555),
    ("305A", "경영대학1호관", "300권역", 37.87005, 127.74510),
    ("305B", "경영대학2호관", "300권역", 37.87035, 127.74560),
    ("306", "중앙도서관", "300권역/교내버스정류장", 37.87025, 127.74435),
    ("307", "농업생명과학대학1호관", "300권역", 37.87115, 127.74550),
    ("308", "농업생명과학대학2호관", "300권역", 37.87155, 127.74595),
    ("309", "농업생명과학대학3호관", "300권역", 37.87185, 127.74635),
    ("310", "영상바이오관", "300권역", 37.87080, 127.74650),
    ("311", "문화예술대학2호관", "300권역", 37.87020, 127.74685),
    ("312", "문화예술대학1호관", "300권역", 37.86975, 127.74655),
    ("314", "태백관(신한은행)", "300권역/복지시설", 37.87070, 127.74490),
    ("315", "한울관(동아리회관)", "300권역", 37.87130, 127.74710),
    ("316", "친환경농업연구센터", "300권역", 37.87220, 127.74680),
    ("317", "농업생명과학대학 신관", "300권역", 37.87145, 127.74570),
    # 400권역
    ("401", "환경연구소", "400권역", 37.86820, 127.74620),
    ("402", "아산관(정보통신연구소)", "400권역", 37.86845, 127.74680),
    ("403", "집현관", "400권역", 37.86880, 127.74715),
    ("404A", "약학대학", "400권역", 37.86910, 127.74750),
    ("404B", "혁신제약임상연구동", "400권역", 37.86925, 127.74780),
    ("405", "창강제지기술연구소", "400권역", 37.86855, 127.74795),
    ("406", "산림환경과학대학2호관", "400권역", 37.86810, 127.74835),
    ("407", "산림환경과학대학1호관", "400권역", 37.86775, 127.74785),
    ("408", "의생명과학대학", "400권역", 37.86730, 127.74850),
    ("409", "동물생명과학대학1호관", "400권역", 37.86795, 127.74920),
    ("410", "동물생명과학대학2호관", "400권역", 37.86835, 127.74965),
    ("411", "수의과대학", "400권역", 37.86750, 127.75010),
    ("412", "동물병원(야생동물구조센터)", "400권역", 37.86715, 127.75055),
    ("413", "의학2호관(간호대학/의과대학)", "400권역", 37.86960, 127.74860),
    ("414", "의학1호관", "400권역", 37.86995, 127.74825),
    ("417", "강원대학교어린이집", "400권역/동문 인근", 37.86690, 127.74720),
    ("418", "스크립스코리아항체연구원", "400권역", 37.86695, 127.74910),
    ("419", "백령스포츠센터", "400권역", 37.86895, 127.74910),
    ("420", "제3학생회관(두리관)", "400권역/두리관", 37.86865, 127.74885),
    ("421", "미래도서관", "400권역/동문권역 도서관", 37.86930, 127.74950),
    ("422", "동물실험센터", "400권역", 37.86980, 127.74905),
    ("423", "평생교육원", "400권역/동문 인근", 37.86655, 127.74810),
    ("424", "목재교육전시장", "400권역", 37.86760, 127.74750),
    ("425", "산학연혁신허브", "400권역", 37.86675, 127.74980),
    ("426", "친환경지류포장소재관", "400권역", 37.86840, 127.74815),
    # 500권역
    ("500", "백령아트센터", "500권역/정문 인근", 37.86665, 127.74295),
    ("501", "서암관(디지털미디어센터)", "500권역", 37.86725, 127.74345),
    ("502", "법학전문대학원", "500권역/법학관", 37.86710, 127.74435),
    ("503", "사회과학대학1호관", "500권역", 37.86765, 127.74390),
    ("509", "인문대학3호관", "500권역", 37.86790, 127.74310),
    ("510", "인문대학2호관", "500권역", 37.86820, 127.74285),
    ("511", "인문대학1호관", "500권역", 37.86805, 127.74355),
    ("512", "첨단건설기술연구소", "500권역", 37.86740, 127.74210),
    ("517", "종합운동장", "500권역/대운동장", 37.86640, 127.74510),
    ("518", "법학도서관", "500권역", 37.86695, 127.74470),
    ("519", "사회과학대학2호관", "500권역", 37.86750, 127.74440),
    ("520", "사범대학부설고등학교", "강원사대부고", 37.87320, 127.74690),
    # 학생생활관(기숙사) 권역
    ("생활관1", "율곡관(학생생활관)", "제1생활관", 37.87120, 127.74830),
    ("생활관2", "퇴계관(학생생활관)", "제1생활관", 37.87150, 127.74860),
    ("생활관3", "다산관(학생생활관)", "제1생활관", 37.87175, 127.74820),
    ("생활관4", "예지원(학생생활관)", "제1생활관", 37.87195, 127.74855),
    ("생활관5", "난지원(학생생활관)", "제2생활관", 37.87055, 127.74890),
    ("생활관6", "국지원(학생생활관)", "제2생활관", 37.87085, 127.74920),
    ("생활관7", "국제생활관", "제3생활관", 37.87260, 127.74780),
    ("생활관8", "국제인재양성관", "제3생활관", 37.87285, 127.74810),
    ("BTL1", "새롬관(제1BTL생활관)", "향토학사 인근", 37.87035, 127.75010),
    ("BTL2", "이룸관(제2BTL생활관)", "동문 기숙사권역", 37.86985, 127.75045),
    ("BTL3", "다온관(제3BTL생활관)", "신축 BTL 기숙사", 37.86945, 127.75070),
]

def fetch_tmap_coords(univ_prefix, building_name, default_lat, default_lon):
    """Tmap POI 검색으로 최신 좌표를 조회하고, 검색 결과가 없으면 캠퍼스맵 기본 좌표를 반환합니다."""
    url = "https://apis.openapi.sk.com/tmap/pois"
    params = {
        "version": 1,
        "searchKeyword": f"{univ_prefix} {building_name}",
        "count": 1
    }
    headers = {"Accept": "application/json", "appKey": TMAP_APP_KEY}
    try:
        res = requests.get(url, params=params, headers=headers, timeout=5)
        if res.status_code == 200 and res.text:
            poi = res.json()['searchPoiInfo']['pois']['poi'][0]
            lat, lon = float(poi['noorLat']), float(poi['noorLon'])
            # 춘천 권역 좌표 범위 검증
            if 37.85 <= lat <= 37.91 and 127.71 <= lon <= 127.78:
                return lat, lon
    except Exception:
        pass
    return default_lat, default_lon

def build_dataframe(univ_name, search_prefix, building_list, use_tmap_api=False):
    rows = []
    for b_no, b_name, note, d_lat, d_lon in building_list:
        if use_tmap_api:
            lat, lon = fetch_tmap_coords(search_prefix, b_name, d_lat, d_lon)
            time.sleep(0.1)  # API 호출 제한 방지
        else:
            lat, lon = d_lat, d_lon
        rows.append({
            "학교": univ_name,
            "건물번호": b_no,
            "건물명": b_name,
            "구명칭_비고": note,
            "위도(lat)": lat,
            "경도(lon)": lon
        })
    return pd.DataFrame(rows)

# CSV 파일 3종 생성 (use_tmap_api=True로 변경 시 Tmap API 실시간 재조회 수행)
df_hallym = build_dataframe("한림대학교", "한림대학교", hallym_buildings, use_tmap_api=False)
df_knu = build_dataframe("강원대학교", "강원대학교 춘천캠퍼스", knu_buildings, use_tmap_api=False)
df_all = pd.concat([df_hallym, df_knu], ignore_index=True)

df_hallym.to_csv(PREP_OUTPUT_DIR / "한림대학교 건물별 좌표.csv", index=False, encoding="utf-8-sig")
df_knu.to_csv(PREP_OUTPUT_DIR / "강원대학교 건물별 좌표.csv", index=False, encoding="utf-8-sig")
df_all.to_csv(PREP_OUTPUT_DIR / "한림대_강원대_캠퍼스맵_통합_좌표.csv", index=False, encoding="utf-8-sig")

print(f"✅ 한림대({len(df_hallym)}개), 강원대({len(df_knu)}개), 통합({len(df_all)}개) CSV 저장 완료!")


✅ 한림대(43개), 강원대(94개), 통합(137개) CSV 저장 완료!


# 버스 노선망 전처리 — TAGO API 3종

이 구역은 **API 주소와 데이터 가공 단계만 준비한 틀**입니다. API 실호출·CSV 저장·사이트 연결은 아직 구현하거나 실행하지 않았습니다.

- 기존 위쪽 7개 셀은 별도 실험 코드입니다. 이 구역을 확인할 때는 **전체 실행 대신 아래 셀만 순서대로 선택**하세요. 위쪽 셀에는 외부 호출과 기존 CSV 저장 코드가 있습니다.
- 작업 순서: **설정 → 수집할 요청 확인 → 원본 응답 보관 → 정규화 → 검증 → 저장 경로 확인**.
- 1차 비교 대상: **한림대 건물 → 맥도날드 춘천후평DT점**, 버스 **300번 / 400번**.
- 아래의 `TODO`는 실제 수집·가공 코드를 작성할 위치입니다. 빈 결과를 검증된 데이터로 취급하지 않습니다.

공식 명세: [버스노선정보](https://www.data.go.kr/data/15098529/openapi.do) · [버스정류소정보](https://www.data.go.kr/data/15098534/openapi.do) · [버스도착정보](https://www.data.go.kr/data/15098530/openapi.do)


## 1. API 주소·인증키·조회 조건

세 API의 공식 기본 주소를 미리 채웠습니다. 승인 화면의 서비스 URL과 비교할 수 있습니다. 아래 요청주소는 **기본 주소 + 기능명**으로 구성합니다.

인증키는 프로젝트 `.env`의 `TAGO_API_KEY`를 우선 읽고, 비어 있으면 기존 `PUBLIC_DATA_KEY`를 사용합니다. 각 항목은 환경변수가 파일 값보다 우선합니다. 기존 공용 키와 별도로 본인 계정의 버스 키를 설정할 수 있습니다. 키를 이 노트북의 문자열이나 출력에 직접 넣지 않습니다. 아래 설정은 기존 셀의 전역변수와 구분하기 위해 `BUS_` 접두사를 사용합니다.


In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import dotenv_values

# 프로젝트 루트 또는 그 하위에서 노트북을 열었을 때 찾습니다.
BUS_PROJECT_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "app.py").is_file()
     and (folder / "data_preprocessing.ipynb").is_file()),
    None,
)
if BUS_PROJECT_ROOT is None:
    raise RuntimeError("hallym_capstone 폴더에서 노트북을 열어 주세요.")

# ★ API 주소 입력·확인 칸: 공공데이터포털의 기본 서비스 URL
BUS_ROUTE_API_URL = "https://apis.data.go.kr/1613000/BusRouteInfoInqireService"
BUS_STOP_API_URL = "https://apis.data.go.kr/1613000/BusSttnInfoInqireService"
BUS_ARRIVAL_API_URL = "https://apis.data.go.kr/1613000/ArvlInfoInqireService"

BUS_API_ENDPOINTS = {
    "route_list": f"{BUS_ROUTE_API_URL}/getRouteNoList",
    "route_detail": f"{BUS_ROUTE_API_URL}/getRouteInfoIem",
    "route_stops": f"{BUS_ROUTE_API_URL}/getRouteAcctoThrghSttnList",
    "stop_search": f"{BUS_STOP_API_URL}/getSttnNoList",
    "stop_routes": f"{BUS_STOP_API_URL}/getSttnThrghRouteList",
    "arrivals": f"{BUS_ARRIVAL_API_URL}/getSttnAcctoArvlPrearngeInfoList",
}

BUS_ENV_FILE = BUS_PROJECT_ROOT / ".env"
BUS_ENV_VALUES = (
    dotenv_values(BUS_ENV_FILE, encoding="utf-8-sig", interpolate=False)
    if BUS_ENV_FILE.is_file() else {}
)
BUS_PUBLIC_SERVICE_KEY = str(
    os.environ.get("PUBLIC_DATA_KEY", BUS_ENV_VALUES.get("PUBLIC_DATA_KEY")) or ""
).strip()
BUS_TAGO_SERVICE_KEY = str(
    os.environ.get("TAGO_API_KEY", BUS_ENV_VALUES.get("TAGO_API_KEY")) or ""
).strip()
BUS_SERVICE_KEY = BUS_TAGO_SERVICE_KEY or BUS_PUBLIC_SERVICE_KEY

BUS_CITY_CODE = "32010"  # 현재 프로젝트의 춘천 코드. 실수집 시 도시코드 목록과 대조
BUS_TARGET_ROUTE_NOS = ["300", "400"]
BUS_CASE = {
    "origin": "한림대학교",
    "destination": "맥도날드 춘천후평DT점",
    "origin_stop_id": "",       # TODO: API로 승차 승강장을 확인한 뒤 입력
    "destination_stop_id": "",  # TODO: API와 지도로 하차 승강장을 확인한 뒤 입력
}

# 다음 단계에서 호출·저장 코드를 작성할 때 사용할 기본값입니다.
# 현재는 스위치를 바꿔도 실제 호출·저장을 수행하는 코드는 없습니다.
BUS_API_CALLS_ENABLED = False
BUS_SAVE_ENABLED = False

print("인증키 설정 여부:", bool(BUS_SERVICE_KEY))
print("비교 노선:", ", ".join(BUS_TARGET_ROUTE_NOS))


## 2. API별 요청 틀

아래 표는 **호출 계획**이며 네트워크 요청을 보내지 않습니다. 인증키 값도 포함하지 않습니다.

| 순서 | 데이터 | 확인할 내용 |
| --- | --- | --- |
| 1 | 노선정보: 번호 목록·상세 | 300·400 검색 결과의 실제 번호, 노선 ID, 기종점 |
| 2 | 노선정보: 경유 정류소 | 선택한 노선 ID의 전체 정류장 순서·상하행 |
| 3 | 정류소정보: 검색·경유노선 | 한림대와 목적지 인근 승강장 ID·좌표·경유 노선 교차검증 |
| 4 | 도착정보 | 확인된 승차 정류장의 실시간 응답 점검. 정적 노선망에는 저장하지 않음 |

목록은 `totalCount`까지 모든 페이지를 받아야 합니다. 상세 단건 조회는 페이지 항목이 없습니다. 요청변수 대소문자도 구분합니다: 노선 조회 `routeId`, 정류소 경유노선 `nodeid`, 도착정보 `nodeId`.


In [ ]:
BUS_LIST_PARAMS = {
    "_type": "json",
    "cityCode": BUS_CITY_CODE,
    "pageNo": 1,
    "numOfRows": 100,
}
BUS_REQUEST_TEMPLATES = {
    "route_list": {**BUS_LIST_PARAMS, "routeNo": "300"},  # 400도 따로 조회
    "route_detail": {"_type": "json", "cityCode": BUS_CITY_CODE, "routeId": ""},
    "route_stops": {**BUS_LIST_PARAMS, "routeId": ""},
    "stop_search": {**BUS_LIST_PARAMS, "nodeNm": "한림대학교"},
    "stop_routes": {**BUS_LIST_PARAMS, "nodeid": ""},
    "arrivals": {**BUS_LIST_PARAMS, "nodeId": ""},
}

bus_request_preview = pd.DataFrame([
    {"기능": name, "요청주소": BUS_API_ENDPOINTS[name], "요청조건": params}
    for name, params in BUS_REQUEST_TEMPLATES.items()
])
bus_request_preview


## 3. 원본 응답 수집 — TODO

다음 단계에서 이 구역에 수집 코드를 작성합니다.

1. 인증키의 인코딩 형태를 확인하고 요청 시 한 번만 적용합니다. 오류 출력에 키가 포함된 URL을 남기지 않습니다.
2. 300·400 노선 목록을 받아 번호가 **정확히 일치하는 항목**만 남깁니다. 같은 번호의 여러 노선 ID는 각각 보존합니다.
3. 공식 번호·기종점을 확인한 뒤 `BUS_SELECTED_ROUTE_IDS`를 채웁니다. 기존 CSV의 ID를 정답으로 고정하지 않습니다.
4. 선택한 노선의 전체 경유 정류소와 한림대·목적지 인근 정류소를 수집합니다.
5. 도착정보는 확인된 승차 정류장에서 별도로 점검합니다. 도착 예정 버스가 없다는 이유로 노선이 없다고 판단하지 않습니다.

아래 배열에는 API 응답의 `items.item` 목록을 넣는 것을 기준으로 합니다. 단일 객체는 목록으로 통일하고, 경유 정류소에는 조회에 사용한 `routeid`를 보존합니다.


In [ ]:
BUS_SELECTED_ROUTE_IDS = []  # TODO: 공식 목록·상세 응답 확인 후 선택

bus_raw_routes = []
bus_raw_route_stops = []
bus_raw_stops = []
bus_raw_stop_routes = []
bus_raw_arrivals = []  # 일시적 점검용. 정적 CSV 저장 대상에서 제외

bus_collection_metadata = {
    "status": "not_collected",
    "city_code": BUS_CITY_CODE,
    "target_route_nos": BUS_TARGET_ROUTE_NOS.copy(),
    "selected_route_ids": [],
    "fetched_at": None,  # TODO: 실제 수집 시 시간대 포함 시각 기록
    "source_urls": BUS_API_ENDPOINTS.copy(),
}

# TODO: BUS_API_CALLS_ENABLED가 True인 경우에만 수집하도록 구현
# TODO: HTTP·인증·응답구조 오류와 페이지 누락·반복을 구분
# TODO: 수집 완료 후 selected_route_ids, fetched_at, status를 갱신


## 4. 정규화할 표 — TODO

아래는 **열 구조가 준비된 빈 표**입니다. 원본을 수집한 뒤 매핑·자료형 변환 코드를 작성합니다.

| 표 | TAGO 원본 → 가공 열 |
| --- | --- |
| 노선 | `routeid → route_id`, `routeno → route_no`, `startnodenm → start_stop_name`, `endnodenm → end_stop_name` |
| 정류소 | `nodeid → stop_id`, `nodeno → stop_no`, `nodenm → stop_name`, `gpslati → lat`, `gpslong → lon` |
| 경유 방문 | `routeid → route_id`, `nodeord → stop_order`, `updowncd → direction_code`, `nodeid → stop_id` |

ID와 버스번호는 문자열로 유지합니다. `direction_code`는 0=상행, 1=하행이며 누락 시 미확인 값으로 남깁니다. 같은 정류장을 여러 번 방문하는 **경유 행은 삭제하지 않습니다**. 방향마다 순번이 다시 시작하는지도 확인하고, 순번을 임의로 다시 매기지 않습니다.


In [ ]:
bus_routes_df = pd.DataFrame(columns=[
    "route_id", "route_no", "start_stop_name", "end_stop_name", "fetched_at",
])
bus_stops_df = pd.DataFrame(columns=[
    "stop_id", "stop_no", "stop_name", "lat", "lon", "fetched_at",
])
bus_route_stops_df = pd.DataFrame(columns=[
    "route_id", "route_no", "direction_code", "stop_order",
    "stop_id", "stop_name", "lat", "lon", "fetched_at",
])

# TODO: bus_raw_*에서 위 스키마로 정규화
# TODO: stop_order 정수, lat/lon 유한한 숫자·춘천 권역 좌표 검사
# TODO: 정류소 master의 동일 ID 이름·좌표 충돌은 기록하고 임의 선택하지 않음
# TODO: 노선 경유 방문은 원래 순번·방향·반복 방문 그대로 보존

pd.DataFrame([
    {"표": "노선", "행 수": len(bus_routes_df)},
    {"표": "정류소", "행 수": len(bus_stops_df)},
    {"표": "경유 방문", "행 수": len(bus_route_stops_df)},
])


## 5. 300·400번 및 후평DT점 사례 검증 — TODO

기존 `data/raw/bus/chuncheon_bus_routes_20260326.csv`는 **비교 대상**입니다. 같은 CSV와 결과가 같다는 이유만으로 실제 운행이 검증된 것으로 처리하지 않습니다.

특히 기존 CSV에서 300번 한림대는 순번 58/58이고, 400번 한림대는 순번 7/57·50/57입니다. 공식 자료의 번호·ID·경유 순서가 어떻게 다른지 기록합니다. 실제 승하차 정류장과 방면은 API 및 지도와 대조해 확정합니다.


In [ ]:
bus_validation_report = {
    "status": "not_run",
    "case": BUS_CASE.copy(),
    "checks": [
        {"id": "route_identity", "내용": "공식 300·400 번호와 ID를 기존 CSV와 대조", "상태": "미실행"},
        {"id": "pagination", "내용": "전체 페이지 수집·누락·반복 여부 확인", "상태": "미실행"},
        {"id": "stop_identity", "내용": "승하차 정류장 ID·좌표와 반대편 승강장 구분", "상태": "미실행"},
        {"id": "visit_order", "내용": "순번 중복·누락 및 반복 방문 보존 확인", "상태": "미실행"},
        {"id": "direction", "내용": "상하행 누락·회차 경계·연속 탑승 여부 확인", "상태": "미실행"},
        {"id": "hupyeong_case", "내용": "한림대→후평DT점 승차 이후 하차 순서·방면 대조", "상태": "미실행"},
    ],
    "warnings": ["수집·가공·검증을 아직 실행하지 않았습니다."],
}

# TODO: 검증 코드를 작성하고 실제 근거에 따라 각 상태·warnings를 갱신
# TODO: 미확인 방향·회차를 검증 완료로 표시하지 않음
pd.DataFrame(bus_validation_report["checks"])


## 6. 저장할 위치 — 경로만 준비

현재는 경로만 선언합니다. 디렉터리·CSV·JSON을 생성하는 코드는 없습니다.

| 예정 파일 | 책임·입력 | 다음 단계의 사용처 |
| --- | --- | --- |
| `data/preparation/generated/tago_bus/bus_routes.csv` | `bus_routes_df`의 공식 노선번호·ID·기종점 | 노선 조회 어댑터 |
| `data/preparation/generated/tago_bus/bus_stops.csv` | `bus_stops_df`의 정류장 ID·좌표 | 주변 정류장 조회 어댑터 |
| `data/preparation/generated/tago_bus/bus_route_stops.csv` | `bus_route_stops_df`의 방향별 경유 방문 | 경로 계산기에 노선망을 공급하는 어댑터 |
| `data/preparation/generated/tago_bus/bus_validation.json` | 수집 출처·시각 및 `bus_validation_report` | 데이터 검토·재가공 기록 |

모든 산출물은 실제 수집·가공·검증 결과에 의존합니다. 도착정보는 정적 파일에 저장하지 않습니다. 기존 원본 CSV는 유지합니다.

현재 사이트의 CSV 로더는 기존 8개 열에 맞춰져 있으므로, 위 파일이 만들어져도 자동 연결되지 않습니다. 서비스 연결과 화면 방면 표시는 별도 작업입니다.


In [ ]:
BUS_OUTPUT_DIR = BUS_PROJECT_ROOT / "data" / "preparation" / "generated" / "tago_bus"
BUS_OUTPUT_PATHS = {
    "routes": BUS_OUTPUT_DIR / "bus_routes.csv",
    "stops": BUS_OUTPUT_DIR / "bus_stops.csv",
    "route_stops": BUS_OUTPUT_DIR / "bus_route_stops.csv",
    "validation": BUS_OUTPUT_DIR / "bus_validation.json",
}

# TODO: 수집·가공·검증 완료 후에만 BUS_SAVE_ENABLED로 저장을 허용
# TODO: CSV는 UTF-8 BOM, index=False; JSON에는 출처·수집시각·미확인 항목 기록
# TODO: 빈 표·검증 실패 자료 저장 및 기존 운영 CSV 덮어쓰기를 막음
# 현재 셀은 mkdir(), to_csv(), 파일 쓰기를 실행하지 않습니다.
